# 03 · Defending in the event data
SkillCorner's dynamic events already describe a lot of defending: on-ball engagements
(pressing, pressure, counter-pressing), pressing chains, defensive phases and line heights.

In [ ]:
import sys
from pathlib import Path

# Make the repo's src/ importable no matter where the notebook is run from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from src.loading import load_all, load_events, load_phases

events = load_all(load_events)
phases = load_all(load_phases)
eng = events[events.event_type == "on_ball_engagement"].copy()
print(f"{len(eng):,} on-ball engagements across {eng.match_id.nunique()} matches")

## Engagement types and what they achieve

In [ ]:
outcomes = ["stop_possession_danger", "reduce_possession_danger", "force_backward",
            "beaten_by_possession", "beaten_by_movement"]
outcomes = [c for c in outcomes if c in eng.columns]
summary = eng.groupby("event_subtype").agg(n=("event_id", "size"),
                                          **{c: (c, "mean") for c in outcomes})
summary.sort_values("n", ascending=False).round(3)

In [ ]:
# Where engagements happen. Thirds appear to be from the engaging (defending) player's
# perspective: pressing happens mostly in their attacking third, pressure in their defensive third.
pd.crosstab(eng.event_subtype, eng.third_start, normalize="index").round(2)

In [ ]:
# Engagements per team per match
per_team = eng.groupby(["match_id", "team_shortname"]).size().groupby("team_shortname").mean()
per_team.sort_values().plot.barh(figsize=(6, 5), title="On-ball engagements per match")
plt.show()

## Pressing chains

In [ ]:
chains = eng.dropna(subset=["pressing_chain_index"]) if "pressing_chain_index" in eng else eng.iloc[0:0]
if len(chains):
    first = chains.drop_duplicates(["match_id", "pressing_chain_index"])
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    first.pressing_chain_length.value_counts().sort_index().plot.bar(ax=ax[0], title="Pressing chain length")
    first.pressing_chain_end_type.value_counts().plot.barh(ax=ax[1], title="How pressing chains end")
    plt.tight_layout()
else:
    print("No pressing chain columns found.")

## Defensive phases
Out-of-possession phase types from the phases of play. How long does each team spend in each
block, and how often do those phases end with the opponent getting a shot?

In [ ]:
blocks = ["high_block", "medium_block", "low_block"]
b = phases[phases.team_out_of_possession_phase_type.isin(blocks)]
(b.groupby("team_out_of_possession_phase_type")
  .agg(phases=("index", "size"), median_seconds=("duration", "median"),
       opponent_shot_rate=("team_possession_lead_to_shot", "mean"),
       opponent_goal_rate=("team_possession_lead_to_goal", "mean"),
       defending_width=("team_out_of_possession_width_start", "median"),
       defending_length=("team_out_of_possession_length_start", "median"))
  .reindex(blocks).round(3))

## Defensive line height
From player-possession events, by the defending team's phase type.

In [ ]:
pp = events[events.event_type == "player_possession"]
col = "last_defensive_line_height_start"
if col in pp:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for t in blocks:
        v = pp.loc[pp.team_out_of_possession_phase_type == t, col].dropna()
        ax.hist(v, bins=40, alpha=0.5, label=f"{t} (n={len(v)})", density=True)
    ax.set_xlabel(col); ax.legend()
    plt.show()
else:
    print(f"{col} not found")

## Pressure on the ball carrier
How often each pressure level leads to a shot, using SkillCorner's 1–5 pressure bands.

In [ ]:
pp.groupby("overall_pressure_start").agg(
    n=("event_id", "size"), lead_to_shot=("lead_to_shot", "mean"),
    epv_for_start=("possession_epv_for_start", "mean")).round(3)